In [1]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import pandas as pd
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')
feature_cols = [
    "Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min",
    "Sensor_S1", "Sensor_S2", "Sensor_S3",
    "S1_missing", "S2_missing", "S3_missing", "S4_missing",
    "is_duplicate_input"]


X = train[feature_cols]
y_class = train["Validity_Label_enc"]
y_reg = train["Reference_Parameter"]

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)


et_f1_scores = []
et_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    clf_et = ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    clf_et.fit(X_tr, y_tr)

    preds = clf_et.predict(X_vl)
    probs = clf_et.predict_proba(X_vl)[:, 1]

    f1 = f1_score(y_vl, preds)
    auc = roc_auc_score(y_vl, probs)
    et_f1_scores.append(f1)
    et_auc_scores.append(auc)

    print(f"Fold {fold}: F1={f1:.4f}, AUC={auc:.4f}")

print(f"\nMean F1 (default threshold): {np.mean(et_f1_scores):.4f} (+/- {np.std(et_f1_scores):.4f})")
print(f"Mean AUC: {np.mean(et_auc_scores):.4f} (+/- {np.std(et_auc_scores):.4f})")

Fold 0: F1=0.6316, AUC=1.0000
Fold 1: F1=0.7727, AUC=1.0000
Fold 2: F1=0.7727, AUC=1.0000
Fold 3: F1=0.6154, AUC=1.0000
Fold 4: F1=0.8261, AUC=1.0000

Mean F1 (default threshold): 0.7237 (+/- 0.0843)
Mean AUC: 1.0000 (+/- 0.0000)


In [2]:
from sklearn.metrics import precision_recall_curve

et_best_f1s = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    clf_et = ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    )
    clf_et.fit(X_tr, y_tr)
    probs = clf_et.predict_proba(X_vl)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_vl, probs)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
    best_idx = f1s.argmax()
    et_best_f1s.append(f1s[best_idx])
    print(f"Fold {fold}: best threshold={thresholds[best_idx]:.3f}, best F1={f1s[best_idx]:.4f}")

print(f"\nMean tuned F1: {np.mean(et_best_f1s):.4f} (+/- {np.std(et_best_f1s):.4f})")

Fold 0: best threshold=0.153, best F1=1.0000
Fold 1: best threshold=0.197, best F1=1.0000
Fold 2: best threshold=0.143, best F1=1.0000
Fold 3: best threshold=0.160, best F1=1.0000
Fold 4: best threshold=0.293, best F1=1.0000

Mean tuned F1: 1.0000 (+/- 0.0000)


In [3]:
from sklearn.ensemble import ExtraTreesRegressor

et_reg_rmse = []
et_reg_mae = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_reg.iloc[train_idx], y_reg.iloc[val_idx]

    reg_et = ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    reg_et.fit(X_tr, y_tr)

    preds = reg_et.predict(X_vl)

    rmse = np.sqrt(mean_squared_error(y_vl, preds))
    mae = mean_absolute_error(y_vl, preds)
    et_reg_rmse.append(rmse)
    et_reg_mae.append(mae)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}")

print(f"\nMean RMSE: {np.mean(et_reg_rmse):.4f} (+/- {np.std(et_reg_rmse):.4f})")
print(f"Mean MAE: {np.mean(et_reg_mae):.4f} (+/- {np.std(et_reg_mae):.4f})")

Fold 0: RMSE=1.0797, MAE=0.6183
Fold 1: RMSE=2.7727, MAE=0.8385
Fold 2: RMSE=1.9890, MAE=0.8123
Fold 3: RMSE=3.3345, MAE=1.0095
Fold 4: RMSE=2.8415, MAE=1.0100

Mean RMSE: 2.4035 (+/- 0.7898)
Mean MAE: 0.8577 (+/- 0.1456)


## Sus Very Perfect Reads

In [5]:
rule_pred = (
    (train["S1_missing"] == 1) |
    (train["S2_missing"] == 1) |
    (train["S3_missing"] == 1) |
    (train["is_duplicate_input"] == 1)
).astype(int)

from sklearn.metrics import f1_score as f1s, accuracy_score

print("Rule-based F1:", f1s(y_class, rule_pred))
print("Rule-based accuracy:", accuracy_score(y_class, rule_pred))

# How many actual Invalids does this rule catch, and how many false positives?
print(pd.crosstab(y_class, rule_pred, rownames=["Actual"], colnames=["Rule Predicted"]))

Rule-based F1: 0.4508670520231214
Rule-based accuracy: 0.905
Rule Predicted    0   1
Actual                 
0               866   0
1                95  39


In [6]:
missed_by_rule = train[(y_class == 1) & (rule_pred == 0)]
print(f"Invalid rows the simple rule misses: {len(missed_by_rule)}")
print(missed_by_rule[["Sensor_S1", "Sensor_S2", "Sensor_S3"]])

Invalid rows the simple rule misses: 95
     Sensor_S1  Sensor_S2  Sensor_S3
9       4.7556    12.2665    17.5003
38     10.8589    10.9508     3.2181
59     23.0025    17.4622    20.4693
63     16.5947    25.0000    21.7478
68     11.1050    17.2072    23.2016
..         ...        ...        ...
974    17.3202     7.0374    17.6327
978    21.8279    24.0598    24.2703
987    10.4687     9.9585    24.1552
992     1.0000    18.8709    24.1812
996    17.7601     8.9330    21.8885

[95 rows x 3 columns]


In [7]:
skf_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

for fold, (train_idx, val_idx) in enumerate(skf_check.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    clf_et = ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    clf_et.fit(X_tr, y_tr)
    probs = clf_et.predict_proba(X_vl)[:, 1]
    preds = clf_et.predict(X_vl)

    print(f"Fold {fold}: F1={f1_score(y_vl, preds):.4f}, AUC={roc_auc_score(y_vl, probs):.4f}")

Fold 0: F1=0.8182, AUC=1.0000
Fold 1: F1=0.7442, AUC=0.9985
Fold 2: F1=0.8000, AUC=1.0000
Fold 3: F1=0.7727, AUC=1.0000
Fold 4: F1=0.7727, AUC=1.0000


## run 3 more seeds on ExtraTrees (10 minutes)

In [8]:
seeds_to_test = [1, 100, 2024]

for seed in seeds_to_test:
    skf_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_f1s = []
    fold_aucs = []

    for train_idx, val_idx in skf_seed.split(X, y_class):
        X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

        clf_et = ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
        )
        clf_et.fit(X_tr, y_tr)
        preds = clf_et.predict(X_vl)
        probs = clf_et.predict_proba(X_vl)[:, 1]

        fold_f1s.append(f1_score(y_vl, preds))
        fold_aucs.append(roc_auc_score(y_vl, probs))

    print(f"Seed {seed}: Mean F1={np.mean(fold_f1s):.4f} (+/- {np.std(fold_f1s):.4f}), Mean AUC={np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")

Seed 1: Mean F1=0.7553 (+/- 0.0774), Mean AUC=0.9997 (+/- 0.0006)
Seed 100: Mean F1=0.7754 (+/- 0.0250), Mean AUC=0.9999 (+/- 0.0002)
Seed 2024: Mean F1=0.7793 (+/- 0.0489), Mean AUC=0.9993 (+/- 0.0007)
